# 23. π0 and FAST — VLA flow policy and action tokenizer

π0 keeps its released structural path at reduced tensor width. FAST keeps the actual DCT/quantization/frequency-major/BPE round trip.


In [ ]:
import math
from collections import Counter

import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(12)
device = torch.device("cpu")


## 1. π0 — 27-layer SigLIP tower, 18 joint layers, GQA, RoPE, block mask, Beta-time flow matching


In [ ]:
class VisionBlock(nn.Module):
    def __init__(self, dim=32, heads=16):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim)
        self.attn = nn.MultiheadAttention(dim, heads, batch_first=True)
        self.norm2 = nn.LayerNorm(dim)
        self.mlp = nn.Sequential(nn.Linear(dim, 4 * dim), nn.GELU(), nn.Linear(4 * dim, dim))

    def forward(self, x):
        q = self.norm1(x)
        y, _ = self.attn(q, q, q, need_weights=False)
        x = x + y
        return x + self.mlp(self.norm2(x))


class SigLIPVision(nn.Module):
    def __init__(self, dim=32, image_size=28, patch=14, depth=27):
        super().__init__()
        self.patch = nn.Conv2d(3, dim, patch, stride=patch)
        count = (image_size // patch) ** 2
        self.position = nn.Parameter(torch.randn(1, count, dim) * 0.02)
        self.blocks = nn.ModuleList([VisionBlock(dim, 16) for _ in range(depth)])
        self.norm = nn.LayerNorm(dim)

    def forward(self, image):
        x = self.patch(image).flatten(2).transpose(1, 2)
        x = x + self.position[:, : x.size(1)]
        for block in self.blocks:
            x = block(x)
        return self.norm(x)


def block_attention_mask(valid, ar_mask):
    ar_mask = ar_mask[None].expand_as(valid)
    groups = torch.cumsum(ar_mask.long(), 1)
    causal = groups[:, None, :] <= groups[:, :, None]
    return causal & valid[:, None, :] & valid[:, :, None]


def pi0_time_embedding(t, dim, min_period=4e-3, max_period=4.0):
    half = dim // 2
    fraction = torch.linspace(0, 1, half, device=t.device)
    period = min_period * (max_period / min_period) ** fraction
    angle = t[:, None] * (2 * math.pi / period)[None]
    return torch.cat([angle.sin(), angle.cos()], -1)


def rope(x, position):
    dim = x.size(-1)
    index = torch.arange(0, dim, 2, device=x.device).float()
    inv = 1.0 / (10000 ** (index / dim))
    angle = position[:, None, :, None].float() * inv[None, None, None]
    even, odd = x[..., 0::2], x[..., 1::2]
    rotated_even = even * angle.cos() - odd * angle.sin()
    rotated_odd = even * angle.sin() + odd * angle.cos()
    return torch.stack([rotated_even, rotated_odd], -1).flatten(-2)


class Pi0Layer(nn.Module):
    def __init__(self, dim=32, q_heads=8, kv_heads=1):
        super().__init__()
        self.q_heads, self.kv_heads = q_heads, kv_heads
        self.head_dim = dim // q_heads
        self.repeat = q_heads // kv_heads
        self.prefix_norm = nn.RMSNorm(dim)
        self.suffix_norm = nn.RMSNorm(dim)
        self.prefix_q = nn.Linear(dim, q_heads * self.head_dim, bias=False)
        self.prefix_kv = nn.Linear(dim, 2 * kv_heads * self.head_dim, bias=False)
        self.suffix_q = nn.Linear(dim, q_heads * self.head_dim, bias=False)
        self.suffix_kv = nn.Linear(dim, 2 * kv_heads * self.head_dim, bias=False)
        self.prefix_out = nn.Linear(dim, dim, bias=False)
        self.suffix_out = nn.Linear(dim, dim, bias=False)
        self.prefix_ffn = nn.Sequential(
            nn.RMSNorm(dim),
            nn.Linear(dim, 4 * dim),
            nn.GELU(),
            nn.Linear(4 * dim, dim),
        )
        self.suffix_ffn = nn.Sequential(
            nn.RMSNorm(dim),
            nn.Linear(dim, 4 * dim),
            nn.GELU(),
            nn.Linear(4 * dim, dim),
        )

    def project(self, x, q_proj, kv_proj, position):
        batch, length, _ = x.shape
        q = q_proj(x).view(batch, length, self.q_heads, self.head_dim).transpose(1, 2)
        kv = kv_proj(x).view(batch, length, 2, self.kv_heads, self.head_dim)
        k, v = kv.permute(2, 0, 3, 1, 4).unbind(0)
        q, k = rope(q, position), rope(k, position)
        return q, k.repeat_interleave(self.repeat, 1), v.repeat_interleave(self.repeat, 1)

    def forward(self, prefix, suffix, mask, position):
        split = prefix.size(1)
        pn, sn = self.prefix_norm(prefix), self.suffix_norm(suffix)
        pq, pk, pv = self.project(pn, self.prefix_q, self.prefix_kv, position[:, :split])
        sq, sk, sv = self.project(sn, self.suffix_q, self.suffix_kv, position[:, split:])
        q, k, v = torch.cat([pq, sq], 2), torch.cat([pk, sk], 2), torch.cat([pv, sv], 2)
        additive = torch.zeros_like(mask, dtype=q.dtype)
        additive = additive.masked_fill(~mask, torch.finfo(q.dtype).min)
        y = F.scaled_dot_product_attention(q, k, v, attn_mask=additive[:, None])
        y = y.transpose(1, 2).contiguous().flatten(2)
        prefix = prefix + self.prefix_out(y[:, :split])
        suffix = suffix + self.suffix_out(y[:, split:])
        return prefix + self.prefix_ffn(prefix), suffix + self.suffix_ffn(suffix)


class Pi0(nn.Module):
    def __init__(self, dim=32, action_dim=3, horizon=4, vocab=64):
        super().__init__()
        self.horizon, self.action_dim, self.dim = horizon, action_dim, dim
        self.vision = SigLIPVision(dim)
        self.language = nn.Embedding(vocab, dim)
        self.state = nn.Linear(action_dim, dim)
        self.action = nn.Linear(action_dim, dim)
        self.action_time = nn.Sequential(nn.Linear(2 * dim, dim), nn.SiLU(), nn.Linear(dim, dim))
        self.layers = nn.ModuleList([Pi0Layer(dim) for _ in range(18)])
        self.out = nn.Linear(dim, action_dim)

    def forward(self, image, language, state, noisy_action, t):
        prefix = torch.cat([self.vision(image), self.language(language)], 1)
        state_token = self.state(state).unsqueeze(1)
        time = pi0_time_embedding(t, self.dim)[:, None].expand(-1, self.horizon, -1)
        action = self.action_time(torch.cat([self.action(noisy_action), time], -1))
        suffix = torch.cat([state_token, action], 1)
        valid = torch.ones(prefix.size(0), prefix.size(1) + suffix.size(1), dtype=torch.bool)
        prefix_ar = torch.zeros(prefix.size(1), dtype=torch.bool)
        suffix_ar = torch.tensor([True, True] + [False] * (self.horizon - 1))
        mask = block_attention_mask(valid, torch.cat([prefix_ar, suffix_ar]))
        position = torch.cumsum(valid.long(), 1) - 1
        for layer in self.layers:
            prefix, suffix = layer(prefix, suffix, mask, position)
        return self.out(suffix[:, -self.horizon:])


pi0 = Pi0()
assert len(pi0.vision.blocks) == 27 and len(pi0.layers) == 18
assert pi0.layers[0].q_heads == 8 and pi0.layers[0].kv_heads == 1
image = torch.randn(1, 3, 28, 28)
language = torch.randint(0, 64, (1, 3))
state = torch.randn(1, 3)
actions = torch.randn(1, 4, 3)
noise = torch.randn_like(actions)
t = torch.distributions.Beta(1.5, 1.0).sample((1,)) * 0.999 + 0.001
x_t = t[:, None, None] * noise + (1 - t[:, None, None]) * actions
F.mse_loss(pi0(image, language, state, x_t, t), noise - actions).backward()


## 2. FAST — normalization, DCT, quantization, frequency-major flattening, BPE round trip


In [ ]:
def dct_matrix(length, device):
    n = torch.arange(length, device=device).float()
    k = torch.arange(length, device=device).float()[:, None]
    matrix = torch.cos(math.pi / length * (n + 0.5) * k)
    matrix[0] *= math.sqrt(1 / length)
    matrix[1:] *= math.sqrt(2 / length)
    return matrix


def normalize_actions(actions, low, high):
    return 2 * (actions - low) / (high - low).clamp_min(1e-6) - 1


def quantize(coefficients, scale=64.0):
    return torch.round(coefficients * scale).long()


def train_bpe(symbols, merges=4):
    vocabulary = [list(symbols)]
    merge_rules = []
    for _ in range(merges):
        sequence = vocabulary[-1]
        pairs = Counter(zip(sequence[:-1], sequence[1:]))
        if not pairs:
            break
        pair, _ = pairs.most_common(1)[0]
        merged_symbol = (pair[0], pair[1])
        new_sequence = []
        index = 0
        while index < len(sequence):
            if index + 1 < len(sequence) and (sequence[index], sequence[index + 1]) == pair:
                new_sequence.append(merged_symbol)
                index += 2
            else:
                new_sequence.append(sequence[index])
                index += 1
        merge_rules.append(pair)
        vocabulary.append(new_sequence)
    return vocabulary[-1], merge_rules


def expand_bpe(tokens):
    output = []
    for token in tokens:
        if isinstance(token, tuple):
            output.extend(expand_bpe(list(token)))
        else:
            output.append(token)
    return output


actions = torch.randn(1, 8, 3)
low = torch.quantile(actions, 0.01, dim=1, keepdim=True)
high = torch.quantile(actions, 0.99, dim=1, keepdim=True)
normalized = normalize_actions(actions, low, high)
coeff = torch.einsum("ft,btd->bfd", dct_matrix(8, actions.device), normalized)
quantized = quantize(coeff)
symbols = quantized.permute(0, 2, 1).reshape(-1).tolist()
bpe_tokens, rules = train_bpe(symbols)
assert expand_bpe(bpe_tokens) == symbols


## Audit result

π0 executes learned SigLIP positions, 8-query/1-KV-head GQA, RoPE, prefix/state/action block attention, Beta(1.5, 1.0) time sampling, and the flow-matching target. FAST decodes the actual merged BPE token stream before reconstruction.
